# Visualizing protein sequence space as a hierarchical cluster plot


This notebook showcases how to analyze a protein family as series of hierarchical clusters.

**Required:** Run the [**run pipeline**](./run_pipeline.ipynb) notebook first (to end of homology method) to make sure the required outputs are present.    

## Step 1. Identify levels of interest and build layout
Can skip if done once and saved.  

In [ ]:
import pickle
from proteinclustertools.layout.circle_layout_tools import Box2DLayout
import re
import os
import pandas as pd

Choose which analysis (homology, kmeans, hierarchical clustering) and which levels to use.   

First list the clustering files in the output folder (could also specify files in 'data_sets' manually):

In [ ]:
data_dir='../output/mmseqs_clustering/mmseqs_clustering_250711/' # could be mmseqs_clustering, kmeans, hierarchical_clustering

# alternative examples for other clustering outputs
# data_dir='../output/kmeans/flattened_10000/' # for kmeans
# data_dir='../output/hierarchical_clustering/' # for hierarchical clustering

data_sets={}
for f in os.listdir(data_dir):
    if f.endswith('.csv'):
        cutoff=f.split('_')[-1][:-4]
        data_sets[cutoff]=data_dir+f

# sort the data sets by cutoff
data_sets={k: data_sets[k] for k in sorted(data_sets, key=lambda x: float(x))}
cutoffs=list(data_sets.keys())
print(cutoffs)

Let's go with the 3 cut-offs corresponding to the 25, 50 and 75 percentiles of bitscores in the example set.

In [ ]:
levels=['100.0', '158.0', '257.0', '410.0']
levels

Load all the data sets (cluster definitions) into one file, with the cut-offs as the column headers.

**Note:** While sequence and cluster IDs are numbers, they are meant to be unique identifiers, and are thus treated as strings by the layout.

In [ ]:
for cutoff in levels:
    clu=pd.read_csv(data_sets[cutoff], dtype=str)
    clu=clu.rename(columns={'cluster':cutoff})
    clu=clu.set_index('id')
    if cutoff==levels[0]:
        clust=clu
    else:
        clust=clust.join(clu, how='outer')
clust.reset_index(inplace=True)
clust.head()

### Build layout, can be skipped if using saved layout

Get a sense of how many clusters should be visualized.

In [ ]:
test_cutoff=5 # minimum number of members in a cluster to be visualized

for level in levels:
    sizes=clust.groupby(level).size()
    print('level', level, ':', sum(sizes>=test_cutoff), 'clusters with at least', test_cutoff, 'members')

Build the layout.

In [ ]:
layout=Box2DLayout(
                    # Minimum required input
                    clust, levels, # which levels to include
                    minsize=test_cutoff, # minimum size of cluster to include
                    
                    # Broad layout options
                    pull=10, # how strongly to pull the clusters together when within the same parent, higher is stronger, usually 1 is fine
                    pull_base=False, radius_pad=5, # for the lowest level with no parents, whether they should be pulled, and how much space to separate them by
                    seed=1, # seed for random number generator, use the same seed to get the same layout

                    # Finetuning the physics simulation
                    cycle_scale=2000, max_cycles=100_000, # how many cycles to run per 50 clusters, up to max_cycles
                    time_step=1/20, velocity_iterations=16, position_iterations=6# finetuning, lower time step and hirher iterations gives nicer packing at the cost of speed
                )

Save layout, especially if it took a while to make.

In [ ]:
# save layout as binary object
with open('layouts/mmseqs_IPR001223_100-158-257-410.pkl', 'wb') as f:
    pickle.dump(layout, f)

# save cluster data as well to avoid remaking, can skip if too large
clust.to_csv('layouts/mmseqs_IPR001223_100-158-257-410.csv', index=False)

## 2. Plot hierarchical cluster plot

In [ ]:
import proteinclustertools.visuals.circle_plot as cp
import proteinclustertools.visuals.annotate as an
import pickle
import pandas as pd
from matplotlib.colors import ListedColormap

Load layout for drawing.

In [ ]:
with open('layouts/mmseqs_IPR001223_100-158-257-410.pkl', 'rb') as f:
    layout=pickle.load(f)

Example of drawing just the layout, with no extra annotation.

In [ ]:
plot, plot_data=cp.CirclePlot(layout, size=500)

#### Work on adding some extra information to the plot.  

**Important note:**
The headers for the fasta file were sanitized into numbers. To match the metadata, each sequence has to be mapped back to its original header.

**Reminder**
Load IDs (sequence or cluster) as strings.

In [ ]:
# load header mapping
header_map=pd.read_csv('../output/IPR001223_FL_header_map.txt', dtype=str)
header_map.head()

Take a look at meta data table (downloaded from Uniprot in this case).

In [ ]:
annot_table=pd.read_csv('../data/IPR001223_bact_arch_jackseq_annot_HMMseqs_and_NOhmmexplodeddomains_ANNOT.tsv',sep='\t')
annot_table.head()

In [ ]:
#fill length column with teh length of the string in column Sequence
annot_table['Length']=annot_table['Sequence'].apply(lambda x: len(x))

Headers in the header_map have to be converted to be same in 'Entry' in the annot_table.

In [ ]:
header_map['Entry|domains|HMM|org']=header_map['header']#.apply(lambda x: x.split('|')[0])
header_map.head()

Load in the cluster definitions and apply the header mapping.

In [ ]:
clusters=pd.read_csv('layouts/mmseqs_IPR001223_100-158-257-410.csv', dtype=str)
levels=clusters.columns[1:]
clusters.head()

In [ ]:
clusters=clusters.merge(header_map[['Entry|domains|HMM|org','id']], on='id')
clusters.head()

#### The two tables now have matching ID columns, we can apply annotations.

In [ ]:
pfamscan = pd.read_csv('../data/converted_protein_domains_pfamscan.tsv', sep='\t')
#change column name domains to pfamscan
pfamscan=pfamscan.rename(columns={'domains':'Pfamscan'})
pfamscan

In [ ]:
annot_table = annot_table.merge(pfamscan[['Entry','Pfamscan']], on='Entry', how='left')
annot_table.to_csv('annot_table_IPR001223_bact_arch_jackseq_annot_HMMseqs_and_NOhmmexplodeddomains_ANNOT_pfamscan.csv', index=False)

In [ ]:
annot_table = pd.read_csv('annot_table_IPR001223_bact_arch_jackseq_annot_HMMseqs_and_NOhmmexplodeddomains_ANNOT_pfamscan.csv', sep=',')

In [ ]:
#calculate the number of values sep by white space in column Pfamscan and add new column with this number for each from called no_pfamscan_domains
annot_table['no_pfamscan_domains'] = annot_table['Pfamscan'].apply(lambda x: len(x.split()) if pd.notna(x) else 0)


In [ ]:
# categorical values
phylums=an.AnnotateClusters(clusters, levels, annot_table, 'phylum', 'Entry|domains|HMM|org', 'Entry|domains|HMM|org')
# numeric values
lengths=an.AnnotateClusters(clusters, levels, annot_table, 'Length', 'Entry|domains|HMM|org', 'Entry|domains|HMM|org', numeric_func='mean')
length_sd=an.AnnotateClusters(clusters, levels, annot_table, 'Length', 'Entry|domains|HMM|org', 'Entry|domains|HMM|org', numeric_func='std')
pfams=an.AnnotateClusters(clusters, levels, annot_table, 'Pfam', 'Entry|domains|HMM|org', 'Entry|domains|HMM|org')
pfamscan=an.AnnotateClusters(clusters, levels, annot_table, 'Pfamscan', 'Entry|domains|HMM|org', 'Entry|domains|HMM|org')
reviewed=an.AnnotateClusters(clusters, [levels[-1]], annot_table, "Reviewed", "Entry|domains|HMM|org", "Entry|domains|HMM|org")
#reps = an.AnnotateClusters(clusters, levels, annot_table, 'Representative', 'Entry|domains|HMM|org', 'Entry|domains|HMM|org')

totaldom=an.AnnotateClusters(clusters, levels, annot_table, 'Total_domains', 'Entry|domains|HMM|org', 'Entry|domains|HMM|org', numeric_func='mean')
no_doms=an.AnnotateClusters(clusters, levels, annot_table, 'no_pfamscan_domains', 'Entry|domains|HMM|org', 'Entry|domains|HMM|org', numeric_func='mean')


In [ ]:
is_reviewed=annot_table['Reviewed']=='reviewed'
reviewed_only=clusters[clusters['Entry|domains|HMM|org'].isin(annot_table[is_reviewed]['Entry|domains|HMM|org'])]
reviewed=an.AnnotateClusters(reviewed_only, [levels[-1]], annot_table, "Reviewed", "Entry|domains|HMM|org", "Entry|domains|HMM|org")
reviewed_colors=an.ColorAnnot(reviewed, cmap=ListedColormap(['red']), saturation=.7)

In [ ]:
#what type of data is in column Pfamscan of annot_table
annot_table['Pfamscan'].value_counts()

In [ ]:
import pandas as pd

# 1. Read your list of “unpromiscuous” PF codes
with open("../data/common_nonprom.txt") as f:
    pf_codes = {line.strip() for line in f if line.strip()}

# 2. Load your annot_table (tsv) into a DataFrame
#annot_table = pd.read_csv("annot_table.tsv", sep="\t")
#ensure Pfamscan column is string
#annot_table = pd.read_csv("annot_table_IPR001223_bact_arch_jackseq_annot_HMMseqs_and_NOhmmexplodeddomains_ANNOT_pfamscan.csv", sep=",")
# Convert Pfamscan column to string type
annot_table["Pfamscan"] = annot_table["Pfamscan"].astype(str)

# 3. Define a boolean mask: does any PF in Pfamscan match one of our pf_codes?
mask = (
    annot_table["Pfamscan"]
    .str.split()                                        # split on whitespace
    .apply(lambda codes: bool(set(codes) & pf_codes))  # intersection non‐empty?
)

# 4. Build com_unprom by selecting the matching rows
com_unprom = annot_table.loc[mask, ["Entry|domains|HMM|org"]].copy()

# 5. Add your new column
com_unprom["common_unprom"] = "CU"

annot_table = annot_table.merge(com_unprom, on="Entry|domains|HMM|org", how="left")
# (Optional) write out to TSV
com_unprom#.to_csv("com_unprom.tsv", sep="\t", index=False)


In [ ]:
# 1. pull out the list of matching entries
cu_entries = com_unprom["Entry|domains|HMM|org"].tolist()

# 2. boolean‐index clusters using that list
reviewed_only = clusters[
    clusters["Entry|domains|HMM|org"].isin(cu_entries)
]

# 3. now annotate and color as before
com_unprom_annot = an.AnnotateClusters(
    reviewed_only,
    [levels[-1]],
    #levels,
    annot_table,
    "common_unprom",
    "Entry|domains|HMM|org",
    "Entry|domains|HMM|org"
)
com_unprom_annot_colors = an.ColorAnnot(
    com_unprom_annot,
    cmap=ListedColormap(['#1E9958']),  # use a single color for unpromiscuous
    saturation=0.7
)


In [ ]:
import pandas as pd

# 1. Read your list of “unpromiscuous” PF codes
with open("../data/common_prom.txt") as f:
    pf_codes = {line.strip() for line in f if line.strip()}

# 2. Load your annot_table (tsv) into a DataFrame
#annot_table = pd.read_csv("annot_table.tsv", sep="\t")
#ensure Pfamscan column is string
#annot_table = pd.read_csv("annot_table_IPR001223_bact_arch_jackseq_annot_HMMseqs_and_NOhmmexplodeddomains_ANNOT_pfamscan.csv", sep=",")
# Convert Pfamscan column to string type
annot_table["Pfamscan"] = annot_table["Pfamscan"].astype(str)

# 3. Define a boolean mask: does any PF in Pfamscan match one of our pf_codes?
mask = (
    annot_table["Pfamscan"]
    .str.split()                                        # split on whitespace
    .apply(lambda codes: bool(set(codes) & pf_codes))  # intersection non‐empty?
)

# 4. Build com_unprom by selecting the matching rows
com_prom = annot_table.loc[mask, ["Entry|domains|HMM|org"]].copy()

# 5. Add your new column
com_prom["common_prom"] = "CP"
annot_table = annot_table.merge(com_prom, on="Entry|domains|HMM|org", how="left")
# (Optional) write out to TSV
com_prom#.to_csv("com_unprom.tsv", sep="\t", index=False)


In [ ]:
# 1. pull out the list of matching entries
cu_entries = com_prom["Entry|domains|HMM|org"].tolist()

# 2. boolean‐index clusters using that list
reviewed_only = clusters[
    clusters["Entry|domains|HMM|org"].isin(cu_entries)
]

# 3. now annotate and color as before
com_prom_annot = an.AnnotateClusters(
    reviewed_only,
    [levels[-1]],
    #levels,
    annot_table,
    "common_prom",
    "Entry|domains|HMM|org",
    "Entry|domains|HMM|org"
)
com_prom_annot_colors = an.ColorAnnot(
    com_prom_annot,
    cmap=ListedColormap(['blue']),
    saturation=0.7
)


In [ ]:
import pandas as pd

# 1. Read your list of “unpromiscuous” PF codes
with open("../data/uncommon_prom.txt") as f:
    pf_codes = {line.strip() for line in f if line.strip()}

# 2. Load your annot_table (tsv) into a DataFrame
#annot_table = pd.read_csv("annot_table.tsv", sep="\t")
#ensure Pfamscan column is string
#annot_table = pd.read_csv("annot_table_IPR001223_bact_arch_jackseq_annot_HMMseqs_and_NOhmmexplodeddomains_ANNOT_pfamscan.csv", sep=",")
# Convert Pfamscan column to string type
annot_table["Pfamscan"] = annot_table["Pfamscan"].astype(str)

# 3. Define a boolean mask: does any PF in Pfamscan match one of our pf_codes?
mask = (
    annot_table["Pfamscan"]
    .str.split()                                        # split on whitespace
    .apply(lambda codes: bool(set(codes) & pf_codes))  # intersection non‐empty?
)

# 4. Build com_unprom by selecting the matching rows
uncom_prom = annot_table.loc[mask, ["Entry|domains|HMM|org"]].copy()

# 5. Add your new column
uncom_prom["uncommon_prom"] = "UP"

# (Optional) write out to TSV
uncom_prom#.to_csv("com_unprom.tsv", sep="\t", index=False)


In [ ]:
# 1. pull out the list of matching entries
cu_entries = uncom_prom["Entry|domains|HMM|org"].tolist()

# 2. boolean‐index clusters using that list
reviewed_only = clusters[
    clusters["Entry|domains|HMM|org"].isin(cu_entries)
]

# 3. now annotate and color as before
uncom_prom_annot = an.AnnotateClusters(
    reviewed_only,
    [levels[-1]],
    #levels,
    annot_table,
    "uncommon_prom",
    "Entry|domains|HMM|org",
    "Entry|domains|HMM|org"
)
uncom_prom_annot_colors = an.ColorAnnot(
    uncom_prom_annot,
    cmap=ListedColormap(['green']),
    saturation=0.7
)


In [ ]:

annot_table = annot_table.merge(uncom_prom, on='Entry|domains|HMM|org', how='left')

In [ ]:
#is_reviewed=annot_table['Reviewed']=='reviewed'
reviewed_only=clusters[clusters['Entry|domains|HMM|org'].isin(annot_table[com_unprom]['Entry|domains|HMM|org'])]
reviewed=an.AnnotateClusters(reviewed_only, [levels[-1]], annot_table, "common_prom", "Entry|domains|HMM|org", "Entry|domains|HMM|org")
reviewed_colors=an.ColorAnnot(reviewed, cmap=ListedColormap(['red']), saturation=.7)

In [ ]:
# 1. pull out the list of matching entries
cu_entries = uncom_prom["Entry|domains|HMM|org"].tolist()

# 2. boolean‐index clusters using that list
reviewed_only = clusters[
    clusters["Entry|domains|HMM|org"].isin(cu_entries)
]

# 3. now annotate and color as before
uncom_prom_annot = an.AnnotateClusters(
    reviewed_only,
    [levels[-1]],
    #levels,
    annot_table,
    "uncommon_prom",
    "Entry|domains|HMM|org",
    "Entry|domains|HMM|org"
)
uncom_prom_annot_colors = an.ColorAnnot(
    reviewed,
    cmap=ListedColormap(['green']),
    saturation=0.7
)


In [ ]:
import pandas as pd
import numpy as np

# 1. Load your promiscuity mapping
prom_df = pd.read_csv("../data/promiscuity copy.txt", dtype=str)  # file has columns Pfam,promiscuity
# create a dict for fast lookup
prom_map = dict(zip(prom_df["Pfam"], prom_df["promiscuos"]))

# 2. Load your annot_table
#annot_table = pd.read_csv("annot_table.tsv", sep="\t", dtype=str)

# 3. Define a function to collect all promiscuity labels for one row
def collect_promiscuity(pfam_cell):
    if pd.isna(pfam_cell):
        return np.nan
    codes = pfam_cell.split()  # assumes whitespace‐separated PF codes
    labs = [prom_map[c] for c in codes if c in prom_map]
    # remove duplicates but keep order
    seen = set()
    uniq = []
    for l in labs:
        if l not in seen:
            seen.add(l)
            uniq.append(l)
    return ";".join(uniq) if uniq else np.nan

# 4. Apply it to make a new column
annot_table["promiscuos"] = annot_table["Pfamscan"].apply(collect_promiscuity)

# 5. (Optional) Save out
#annot_table#.to_csv("annot_table_with_promiscuity.tsv", sep="\t", index=False)


In [ ]:
annot_table .columns

In [ ]:
#unique values in promiscuos column
annot_table["promiscuos"].unique()

In [ ]:
#prom_map_df is annot_table with promiscuity column and "Entry|domains|HMM|org" column but remove rows with Nan in promiscuity column
prom_map_df = annot_table[annot_table["promiscuos"].notna()][["Entry|domains|HMM|org", "promiscuos"]].copy()
#replace CN;CP with CP;CN, replace NP;CP with CP;NP, replace NP;CN with CN;NP
prom_map_df["promiscuos"] = prom_map_df["promiscuos"].str.replace("CN;CP", "CP;CN")
prom_map_df["promiscuos"] = prom_map_df["promiscuos"].str.replace("NP;CP", "CP;NP")
prom_map_df["promiscuos"] = prom_map_df["promiscuos"].str.replace("NP;CN", "CN;NP")

# normalize in-place on annot_table
annot_table["promiscuos"] = (
    annot_table["promiscuos"]
      .str.replace("CN;CP", "CP;CN", regex=False)
      .str.replace("NP;CP", "CP;NP", regex=False)
      .str.replace("NP;CN", "CN;NP", regex=False)
)

#print( annot_table["promiscuity"].unique() )


#prom_map_df.head()
# 1. pull out the list of matching entries
cu_entries = prom_map_df["Entry|domains|HMM|org"].tolist()

# 2. boolean‐index clusters using that list
reviewed_only = clusters[
    clusters["Entry|domains|HMM|org"].isin(cu_entries)
]

# 3. now annotate and color as before
reviewed = an.AnnotateClusters(
    reviewed_only,
    [levels[-1]],
    #levels,
    annot_table,
    "promiscuos",
    "Entry|domains|HMM|org",
    "Entry|domains|HMM|org"
)
reviewed_colors = an.ColorAnnot(
    reviewed,
    cmap=purple_cmap,  # reversed purple colormap
    saturation=0.7
)


In [ ]:
#display unique values in promiscuity column
unique_promiscuity = annot_table["promiscuos"].unique()
print(unique_promiscuity)

In [ ]:
prom_colors = ["#2F0133",  # pastel red
    "#6F2975",  # pastel green
    

    "#C58ACA",  # pastel yellow
    "#6666FF",  # pastel blue

    "#FF66FF",  # pastel magenta
    "#66FFFF",  # pastel cyan
    ]
my_cmap = ListedColormap(prom_colors, name='prom_custom')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

three_purples = [
    #"#2E0854",  # dark
    #"#800080",  # medium
      # light
    "#BEBADA",
    "#585184",  # medium
    "#1F1B3C"  # dark
]

# Create and (optionally) register the colormap
purple_cmap = ListedColormap(three_purples, name="three_purples")
plt.register_cmap(cmap=purple_cmap)


In [ ]:
from matplotlib.colors import ListedColormap

# 1) Build prom_map_df with only the rows that have a promiscuity label
prom_map_df = annot_table.loc[
    annot_table["promiscuos"].notna(),
    ["Entry|domains|HMM|org", "promiscuos"]
].copy()
#loop through the promiscuity column and replace the values CN;CP with CP;CN
prom_map_df["promiscuos"] = prom_map_df["promiscuos"].apply(
    lambda x: ";".join(sorted(x.split(";"))) if pd.notna(x) else x
)

#replace CN;NP with NP;CN and CP;NP with NP;CP
prom_map_df["promiscuos"] = prom_map_df["promiscuos"].apply(
    lambda x: x.replace("CN;NP", "NP;CN").replace("CP;NP", "NP;CP") if pd.notna(x) else x
)    

# normalize in-place on annot_table
annot_table["promiscuos"] = (
    annot_table["promiscuos"]
      .str.replace("CN;CP", "CP;CN", regex=False)
      .str.replace("NP;CP", "CP;NP", regex=False)
      .str.replace("NP;CN", "CN;NP", regex=False)
)
# 1. pull out the list of matching entries
cu_entries = prom_map_df["Entry|domains|HMM|org"].tolist()

# 2. boolean‐index clusters using that list
reviewed_only = clusters[
    clusters["Entry|domains|HMM|org"].isin(cu_entries)
]

# 3. now annotate and color as before
prom_only = an.AnnotateClusters(
    reviewed_only,
    [levels[-1]],
    #levels,
    annot_table,
    "promiscuos",
    "Entry|domains|HMM|org",
    "Entry|domains|HMM|org"
)
prom_only_colors = an.ColorAnnot(
    promsct,
    cmap=no_white_purples,
    saturation=0.7
)

In [ ]:
#unique id values in prom_only
unique_ids = prom_only["promiscuos"].unique()
print(unique_ids)

In [ ]:
# Can also merge columns
has_pdb=annot_table['PDB'].notnull()
is_reviewed=annot_table['Reviewed']=='reviewed'
# select rows with reviewed
reviewed=annot_table[is_reviewed]['Entry'].to_frame()
reviewed['Status']='Reviewed'
# select rows with PDB
pdb=annot_table[has_pdb]['Entry'].to_frame()
pdb['Status']='PDB'
# concatenate the two
status_table=pd.concat([reviewed, pdb])
status_table.head()

In [ ]:
# annotate just the highest level
status=an.AnnotateClusters(clusters, [levels[-1]], reps, 'Representative', '', 'Entry')

Annotations only map the information. To display it, we need to apply colors.

In [ ]:
phylum_colors=an.ColorAnnot(phylums, cmap='Set3', top_n=10, saturation=.7, shuffle_colors_seed=1)
length_colors=an.ColorAnnot(lengths, cmap='viridis', saturation=.7, vmin=300, vmax=1000)
length_sd_colors=an.ColorAnnot(length_sd, cmap='Blues', saturation=.7, vmin=0, vmax=100)
pfams_colors=an.ColorAnnot(pfams, cmap='Set3', top_n=20, saturation=.7, shuffle_colors_seed=1)
pfamscan_colors=an.ColorAnnot(pfamscan, cmap='tab20c', top_n=20, saturation=.7, shuffle_colors_seed=1)

totaldoms_colors=an.ColorAnnot(totaldom, cmap='viridis', saturation=.7, vmin=1, vmax=3)
no_doms_colors=an.ColorAnnot(no_doms, cmap='viridis', saturation=.7, vmin=1, vmax=5)
reviewed_colors=an.ColorAnnot(reviewed, cmap=ListedColormap(['black','red']), saturation=.7)
prom_dom_colors=an.ColorAnnot(prom_dom, cmap='Set3', top_n=20, saturation=.7, shuffle_colors_seed=1)
#reps_colors=an.ColorAnnot(reps, cmap=ListedColormap(['red','yellow']), saturation=.7, binary_blend=True)
#status_colors=an.ColorAnnot(status, cmap=ListedColormap(['red','yellow']), saturation=.7, binary_blend=True)

**Note: To specify the color order** (as opposed to automatically assigned), use the 'annot_order' option. Give a list of annotation values that match the order of the 'cmap'. Works with both built-in and custom ListedColormaps.

In [ ]:
status_colors=an.ColorAnnot(status, cmap=ListedColormap(['red','yellow']), saturation=.7, binary_blend=True, annot_order=['Reviewed','PDB'])

Categorical visualizations.

In [ ]:
# plot with color
plot, plot_data3=cp.CirclePlot(layout, size=800, 
                              annot_text=[pfamscan, com_unprom_annot], annot_top_n=15, annot_na=False,
                              annot_colors=pfamscan_colors)

In [ ]:
# plot with color
plot, plot_data3=cp.CirclePlot(layout, size=800, 
                              annot_text=[pfamscan, com_unprom_annot], annot_top_n=15, annot_na=False,
                              annot_colors=pfamscan_colors, 
                              outlines=com_unprom_annot_colors, highlight_line_width=2)

In [ ]:
# plot with color
plot, plot_data4=cp.CirclePlot(layout, size=800, 
                              annot_text=[pfamscan, com_prom_annot], annot_top_n=15, annot_na=False,
                              annot_colors=pfamscan_colors, 
                              outlines=com_prom_annot_colors, highlight_line_width=2)

In [ ]:
# plot with color
plot, plot_data5=cp.CirclePlot(layout, size=800, 
                              annot_text=[pfamscan, uncom_prom_annot], annot_top_n=15, annot_na=False,
                              annot_colors=pfamscan_colors, 
                              outlines=uncom_prom_annot_colors, highlight_line_width=2)

In [ ]:
# plot with color
plot, plot_data6=cp.CirclePlot(layout, size=800, 
                              annot_text=[reviewed,com_unprom_annot], annot_top_n=15, annot_na=False,
                              annot_colors=reviewed_colors,
                                outlines=com_unprom_annot_colors,
                               highlight_line_width=2)

In [ ]:
# plot with color
plot, plot_data3=cp.CirclePlot(layout, size=800, 
                              annot_text=[pfams, lengths], annot_top_n=15, annot_na=False,
                              annot_colors=pfams_colors, 
                              outlines=length_colors, highlight_line_width=1)

In [ ]:
# plot with color
plot, plot_data2=cp.CirclePlot(layout, size=800, 
                              annot_text=[pfamscan, no_doms], annot_top_n=15, annot_na=False,
                              annot_colors=pfamscan_colors, outlines=no_doms_colors,
                               highlight_line_width=2)

In [ ]:
# plot with color
plot, plot_data2=cp.CirclePlot(layout, size=800, 
                              annot_text=[pfamscan], annot_top_n=15, annot_na=False,
                              annot_colors=pfamscan_colors, 
                               highlight_line_width=3)

Numeric visualizations.

In [ ]:
# plot numeric values
plot, plot_data2=cp.CirclePlot(layout, size=800, 
                              annot_text=[status],
                              annot_colors=status_colors)#, outlines=length_sd_colors)

Mix of both.

In [ ]:
plot, plot_data3=cp.CirclePlot(layout, size=800,
                                annot_text=[phylums, lengths],
                                annot_colors=phylum_colors, 
                                outlines=length_colors)

Make a plot for saving and export using Matplotlib.

In [ ]:
fig=cp.ExportFigure(layout, plot_data3, size=8)

In [ ]:
fig.savefig('figures/mmseqs_IPR001223_pfamscan_tab20c.pdf', bbox_inches='tight')

## 3. Adding annotation tables  
When making the hierarchical cluster plot, we can add a data table that displays certain data for clusters.

We can re-use some of the data we've made.

First, the table we want to show (needs to have a "key" column that's used to ID the sequence). In this case we will use "Entry" as the key column.

In [ ]:
annot_table.head() # table to display on click

Next, a mapping table that matches each ID ("Entry" in this case) to its corresponding cluster in each level.

In [ ]:
clusters.head() # table for mapping each Entry to its level

Finally, we supply the table arguments to the circle plot function. Now clicking on the clusters fills the table with data. Note the data does not get compressed, and large tables (>10Mb) should be loaded at user's own discretion. It is best to label just what the user needs.

In [ ]:
plot_table, plot_table_data = cp.CirclePlot(    layout,
                                                    annot_text=[pfams, reps],
                                                    annot_colors = pfams_colors,
                                                    outlines=reps_colors,
                                                    # data table arguments
                                                    data_on_select_table=annot_table,
                                                    mapping_table=clusters,
                                                    data_key_column='Entry|domains|org'
                                            )

## Saving as HTML

In addition to producing figures for publication, one can store an interactive version for later use or sharing. The HTML version can be opened in a browser (e.g., Chrome)

In [ ]:
from bokeh.plotting import output_file, save

output_file('figures/mmseqs_IPR001223_133-205-285_pfam_reps.html')
save(plot_table)